In [32]:
import numpy as np
from PIL import Image, ImageDraw
from numba import njit, prange

def load_image(image_path):
    """Загрузка изображения."""
    img = Image.open(image_path).convert('L')
    return np.array(img, dtype=np.int32)

@njit(parallel=True, fastmath=True)
def compute_corner_response(image, window_size, offsets):
    """Вычисление corner response."""
    height, width = image.shape
    half = window_size // 2
    corner_response = np.zeros((height, width), dtype=np.float32)
    
    for y in prange(half, height - half):
        for x in prange(half, width - half):
            min_variation = np.inf
            for dx, dy in offsets:
                variation = 0.0
                for wy in range(-half, half + 1):
                    for wx in range(-half, half + 1):
                        x_shifted = x + wx + dx
                        y_shifted = y + wy + dy
                        if 0 <= x_shifted < width and 0 <= y_shifted < height:
                            diff = image[y + wy, x + wx] - image[y_shifted, x_shifted]
                            variation += diff * diff
                if variation < min_variation:
                    min_variation = variation
            corner_response[y, x] = min_variation
    return corner_response

def moravec_corner_detector(image, window_size=3, threshold=1000, nms_radius=3):
    """Детектор Моравека с точной визуализацией крестов."""
    offsets = np.array([(1, 0), (1, 1), (0, 1), (-1, 1),
                       (-1, 0), (-1, -1), (0, -1), (1, -1)], dtype=np.int32)
    
    corner_response = compute_corner_response(image, window_size, offsets)
    threshold_mask = corner_response > threshold
    
    corners = []
    for y in range(nms_radius, corner_response.shape[0] - nms_radius):
        for x in range(nms_radius, corner_response.shape[1] - nms_radius):
            if not threshold_mask[y, x]:
                continue
            local_patch = corner_response[y-nms_radius:y+nms_radius+1, 
                                         x-nms_radius:x+nms_radius+1]
            if corner_response[y, x] == np.max(local_patch):
                corners.append((x, y)) 
    
    # Визуализация
    marked_image = Image.fromarray(image.astype(np.uint8)).convert('RGB')
    draw = ImageDraw.Draw(marked_image)
    cross_size = 3  # Длина "лучей" креста

    for x, y in corners:
        # Горизонтальная линия (красная)
        draw.line([x - cross_size, y, x + cross_size, y], fill='red', width=1)
        # Вертикальная линия (красная)
        draw.line([x, y - cross_size, x, y + cross_size], fill='red', width=1)
    
    return corners, marked_image

In [ ]:
image_name = "g"

input_path = f"origins/{image_name}.jpg"
output_path = f"results/task_1/{image_name}_corners.jpg"
window_size = 20
threshold=7000
nms_radius=10

image = load_image(input_path)

corners, marked_image = moravec_corner_detector(
    image, 
    window_size, 
    threshold, 
    nms_radius
)

marked_image.show() 
marked_image.save(output_path) 